# A ResNet bottleneck block as an nn.Module

Deep ResNets use a **bottleneck** block: a 1x1 conv shrinks the channels, a 3x3 conv works at the narrow width, and a 1x1 conv expands back. Write it as an `nn.Module`, load given weights through a `state_dict`, and run it in eval mode.

## The module
Write `class Bottleneck(nn.Module)` with `__init__(self, c_in, c_mid, c_out, stride)` and these submodules (names matter, they are the `state_dict` keys):

| name | layer |
|---|---|
| `conv1` | `Conv2d(c_in, c_mid, 1, bias=False)` |
| `bn1` | `BatchNorm2d(c_mid)` |
| `conv2` | `Conv2d(c_mid, c_mid, 3, stride=stride, padding=1, bias=False)` |
| `bn2` | `BatchNorm2d(c_mid)` |
| `conv3` | `Conv2d(c_mid, c_out, 1, bias=False)` |
| `bn3` | `BatchNorm2d(c_out)` |
| `downsample` | only if `stride != 1` or `c_in != c_out`: `Sequential(Conv2d(c_in, c_out, 1, stride=stride, bias=False), BatchNorm2d(c_out))`; otherwise `None` |

Forward: $y = \text{ReLU}\big(\text{bn3}(\text{conv3}(\text{ReLU}(\text{bn2}(\text{conv2}(\text{ReLU}(\text{bn1}(\text{conv1}(x)))))))) + s(x)\big)$, where $s$ is `downsample` or the identity. The stride sits on the 3x3 conv.

## The function
Write `bottleneck_forward(x, w1, bn1, w2, bn2, w3, bn3, wd, bnd, stride)`:

- `x`: float64 `(N, c_in, H, W)`.
- `w1`, `w2`, `w3`: conv weights in `nn.Conv2d` layout; their shapes give `c_in`, `c_mid`, `c_out`.
- `bn1`, `bn2`, `bn3`: float64 `(4, C)`, rows `[weight, bias, running_mean, running_var]` of the matching BatchNorm.
- `wd`, `bnd`: the downsample conv weight `(c_out, c_in, 1, 1)` and its BatchNorm `(4, c_out)`, or both `None` when the block has no downsample.
- `stride`: int.

Build the module in float64, load the weights with `load_state_dict` (all keys, strict), switch to **eval** mode (BatchNorm uses the running statistics, `eps = 1e-5`), and return a tuple `(y, n_params)`: the output and the number of trainable parameters (`sum of p.numel()` over `parameters()`, an int).

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
from torch import nn

class Bottleneck(nn.Module):
    def __init__(self, c_in, c_mid, c_out, stride):
        super().__init__()
        # TODO: conv1/bn1, conv2/bn2, conv3/bn3 and an optional downsample
        self.downsample = None

    def forward(self, x):
        raise NotImplementedError


def bottleneck_forward(x, w1, bn1, w2, bn2, w3, bn3, wd, bnd, stride):
    """Return (y, n_params)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[-1.5, -0.5], [0.5, 1.5]]]], "dtype": "float64"}, {"__nd__": [[[[1.0]]]], "dtype": "float64"}, {"__nd__": [[1.0], [0.0], [0.0], [1.0]], "dtype": "float64"}, {"__nd__": [[[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[1.0], [0.0], [0.0], [1.0]], "dtype": "float64"}, {"__nd__": [[[[1.0]]]], "dtype": "float64"}, {"__nd__": [[1.0], [0.0], [0.0], [1.0]], "dtype": "float64"}, null, null, 1], "e": {"__tup__": [{"__nd__": [[[[0.0, 0.0], [0.5, 1.5]]]], "dtype": "float64"}, 17]}}, {"a": [{"__nd__": [[[[1.0, 1.0], [1.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0]]]], "dtype": "float64"}, {"__nd__": [[1.0], [0.0], [0.0], [1.0]], "dtype": "float64"}, {"__nd__": [[[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[1.0], [0.0], [0.0], [1.0]], "dtype": "float64"}, {"__nd__": [[[[1.0]]], [[[1.0]]]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0], [0.0, 0.0], [0.0, 0.0], [1.0, 1.0]], "dtype": "float64"}, {"__nd__": [[[[1.0]]], [[[-1.0]]]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0], [0.0, 0.0], [0.0, 0.0], [1.0, 1.0]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[[[4.999935000787492]], [[2.9999450007124926]]]], "dtype": "float64"}, 26]}}, {"a": [{"__nd__": [[[[0.8999031001263527, -0.4583752814182017, -1.3140853562711352, 0.6586976762683966, 0.9554258589047299], [0.5246310370122538, -2.5582694047689714, -1.5353412369219648, 0.6136306992844481, 1.4943659536833518], [-1.4945822826806263, 0.9703254902570809, 1.119085194938211, 0.017410577559073888, -1.3391334258272456], [-0.6403135502871974, -0.36200038995477235, 0.6909023107993615, 0.8993356142445694, -0.9868487544980165], [-0.10213500978610211, -0.03492662101325654, -0.1748647645292447, 0.9459331720697963, -0.2716423526275381]], [[-0.07001312209709928, -1.4837713985510979, 0.1461555470109563, -0.9387549854076984, 1.8856044290505742], [0.14862462571017082, 0.18387348762022965, 0.09867302218535816, -1.1710098365424229, -0.4353877520140602], [0.11089904774551591, 0.9117033920612675, 0.21966463374212455, -1.322074441212135, -0.21426067432006896], [-0.1381618504679183, -1.6847881525169563, 0.5395525647847532, 0.7637683732732645, -0.9124293751946004], [2.1853321886520645, 0.3248373208916849, -0.5884791686264779, 1.9458446627391843, -0.5886527815668982]], [[0.14952750319362904, -1.8125045765170995, -1.097822920126895, 1.7502511127769047, -1.0016317993026647], [-0.8418653938708923, -0.2603692595885055, -0.8247355444453578, -1.2520311039164058, -1.022078445411505], [-0.17003100952164193, 1.51313095370942, 0.7696735500412056, 2.2639701559172947, -0.8698100831646507], [0.1731788777685994, 0.25686518788175855, 0.5076765842954212, 0.7218186857154646, -1.7291629284996182], [1.475976388597113, -0.013240644926565882, -0.497206650841798, 0.2535164485020698, -0.5871797011832962]], [[-1.2591904024776606, -0.37412478772866653, -1.9415563597848238, -1.1277701703186074, 0.7236483717693785], [0.47564477846306924, -1.1510973312881585, -2.245474992652233, 0.1750205685566896, 0.3698271416967061], [-0.05980798561477236, -1.001621754120705, 0.05884238902693108, -0.3662016246466834, 0.18242126882969942], [0.5660898440135426, -0.046660454122719186, -0.2123861498342949, 0.5310394307239042, 0.2601905067957917], [-0.8776058908998275, -0.1026525948220941, 0.6619710401165064, 0.566676665955915, -0.3249473885005216]], [[-0.16475776319822014, -1.054644076054288, 0.22098209994482518, 1.0684579039156317, 0.40711414962432146], [-0.4675931693904739, 0.6242400827470025, -0.004906778821869204, 0.63114207672625, -1.2471394132653437], [-0.416740309187273, 0.7652712750397983, 1.1058533299740014, -0.7801048656528086, -0.501994535936666], [1.8491978579821637, 0.36501959777152776, -0.15312121222425223, 0.48895736361440245, -0.5500261650652837], [-1.6647868040313025, 0.026143143440907034, -0.34950288107148486, -0.6110222864556403, 0.007791154213793635]], [[-0.8798443118302758, -0.49356126390405336, -1.8657893680378774, 1.1923428610079017, 1.424031771626747], [-1.1406864833252963, -1.5961690163759694, 0.38784548668602986, 0.7894242575939637, -0.45914222326017917], [-1.1156907109095366, 1.832357245414387, -0.6859242727002945, 0.369590973978388, 0.28819304389267986], [0.3097412747127935, 1.2666064149008072, 1.3789180429319672, -1.4739190204607706, -0.1256207111560749], [1.0913655263956208, 0.17375718881896016, -0.6511379645003832, 0.7755192862785163, -0.5232162470079974]], [[-0.18898096961976896, 0.8250610989248086, 0.5654955393229438, 1.880777388944944, 0.6875735523594396], [0.003970490450789027, 1.0332063457602467, -0.04454539543872009, 0.06852097081902243, 0.13934197549212315], [-0.41248008364767336, 0.2647185027449832, 0.16663350972278246, 0.1712744563733538, -0.9556545456354959], [0.5732484727593253, 0.10141777533674344, -0.15393569734150708, -0.14865357036849428, -0.6692723594336016], [-0.6993617502811615, 1.8623509662854476, 0.17674580120788055, 0.7380907943308884, 0.6799888584762429]], [[-0.7537494715552596, -0.1489208632662329, -0.5416708517174382, 0.5178246521935177, 0.3550922051871897], [-0.9670685087806554, 0.9322241979863367, 0.9030817092375732, -1.390395051965165, 0.36071344157505386], [0.2591470004437754, -1.7361066262995686, -1.1600363289123228, 0.5330931653127619, -1.2110098525158735], [-1.953915228267994, 0.03853512001323975, -1.965506840216971, -0.2067981364201119, 0.09442973617892346], [-0.10538600761714678, 0.5674481968126461, -0.08169069589274013, -0.28349109229612424, -1.143373383054588]]], [[[1.1357826572659628, -1.1710913756748704, 0.34450303703876567, -0.09653632348070056, -0.42605814863482355], [-1.3153216324715014, -1.6966569050803686, -0.09251436450625876, 0.6657434056871472, 0.05454370584463922], [-1.1645616900725615, 0.9531631679323253, -2.605194551392579, -0.13056007303774697, 0.9154669818295245], [1.034753546064471, -0.1462986616634759, 1.2095682219216541, 1.7283620189763906, -0.2734018292485807], [1.104597836294156, 0.06376874964000213, 1.0828369944733724, -1.1844961746628253, 0.7466009183929868]], [[1.2563725005601765, 1.1375545801968048, 0.09971880935193786, -1.9823657376675852, 0.28619047869206626], [-2.3182586446678246, 0.14128269177945466, 0.20135077984155927, 0.6472941471980846, -0.5949511424682283], [0.687702037620633, -0.33575220233473985, -1.1458566901571092, 0.17348984634428655, 2.262974698580738], [0.35219986567768247, 0.47316689424516145, 1.6408906014381224, 1.104612494582175, 1.1021847358732486], [-1.4408499438047204, -0.7911708487818564, 2.235787902369628, 0.22245862678087544, -0.8280426525904498]], [[1.2612530489294855, 0.3924678955072567, -1.929319304805443, -0.6627941102870671, -1.1360649762151427], [-1.2008288643928913, -0.7337416134191422, -1.3539620228748437, -0.4509166665044499, 0.2368925526421099], [-0.2071022825230599, -0.21022496764490686, 0.9449697475420566, -0.297468675038525, 0.1296825277507437], [-0.8770400487832429, -1.186238907029351, 0.5110009184237779, -0.9727982223567224, 0.13335036080827176], [0.8983230160832718, -0.4918181534485319, 2.117896393050123, 1.154932989517064, -0.7749416494499567]], [[2.132305632290582, 0.512600700683071, 1.0842838741891545, -0.4711441007214935, 0.3590772814392224], [-0.23570897450726722, 0.32787796947889997, -0.6708052031698299, -0.48719840283798577, -0.7413039433283569], [-2.2100265053547945, -0.863276535734303, 0.27013428810193063, -1.2500737283156964, -1.0433697266285642], [1.3556078541170875, -1.7961822163322498, -0.24617260495413848, 0.4725686239897306, 0.6061264857762885], [0.3723907825281214, 0.28770832323432133, -0.07896576457099899, -1.1026699520104142, -2.212771329729929]], [[1.6808733409243415, -0.4194087775526541, 0.10577099091471026, -1.0259913075104412, 0.022249902628969653], [-1.4284526835500029, -0.8158507682577147, 0.19532311375772385, 1.7706805662541005, 0.4996791666235676], [2.0179903702138553, 0.27029886190602875, 0.20965174710318787, 1.453531663956659, 0.01267213663205362], [-1.480634639858278, 0.5149244150457037, 0.39886576124664397, 0.18802908023473097, 0.45602163855720473], [-0.3406609736287932, -0.08864978070318917, -1.288060151152161, 0.7480973399644907, -0.6841589749282385]], [[1.2231075472760802, 0.1002706481013557, -0.8516147327509392, 0.697455053954254, 0.35997953593166326], [0.43590421830446396, 0.05157991708721518, -0.5600207854746914, 1.49609223124166, -1.5523928703565835], [-1.0410390314277127, 1.2759384519437658, 1.7062774598508694, -1.929321285825481, 1.7610541624095817], [0.6014498790332848, -2.006201664829436, -0.6632086761983189, -1.1056886172026479, 0.07043773637851994], [-0.2917822530201587, 1.2332061400928047, 0.1874577704565584, 0.8555373259936064, 0.9974476837664051]], [[-1.1049767778380635, -0.14799921672887872, 0.4785978382471575, 0.2995858018609027, -0.6577840778967441], [0.4879797017976916, -1.1764970763216078, -0.6216272988342539, 0.5223340397982632, 0.6654309691428967], [-1.0732537736646675, 0.21599442835453306, 0.5570297040139263, 0.9360799502069432, 0.6457073040005146], [0.1640362741793667, -1.0509146961121778, -1.6835135692212928, -0.857265544666277, -0.6963451411130358], [-0.45553713139695384, 0.2534519869092087, 0.5250223396965206, 1.9980356141588071, -1.6953503102583196]], [[-0.9134610178336446, -1.7069451973431744, -1.1620582955953451, -1.4462888262578857, -0.935226281377931], [-0.9528447470150309, -1.2973151299040195, -0.7494124870229606, 0.6836749752991402, 0.6057178944027789], [-0.15176734784717566, 0.7778471371427439, -2.188178187401177, -1.06125201967092, 0.74381545865822], [-0.44453996760106884, -0.6817123760204092, -0.45675763978006695, -0.4625297791596233, 0.704009483093989], [1.4472446555312715, -1.3964547369467992, -1.1432618612716308, -0.6047203051694087, 0.2387735564963027]]]], "dtype": "float64"}, {"__nd__": [[[[0.24332470289545088]], [[-0.031026984239738188]], [[0.13552319467702478]], [[-0.10797115499768145]], [[0.1309206124683812]], [[-0.7332259875149292]], [[0.2517635931927977]], [[0.01592670323952682]]], [[[-0.4058398727505617]], [[-0.023969389245911597]], [[-0.44636274588235747]], [[0.39205745859378205]], [[0.3794691590510492]], [[0.02998488021321906]], [[-0.6190776333822791]], [[-0.7297284358620149]]]], "dtype": "float64"}, {"__nd__": [[0.6367920081257327, 1.0392292456127143], [0.20426648269539813, -0.15072512507884123], [-0.31560294076303247, 0.1836920271731553], [1.5790192666693164, 1.9547338671203043]], "dtype": "float64"}, {"__nd__": [[[[0.1894555667632322, 0.09047287554093388, -0.10201807238745822], [-0.3284235346997423, -0.17877512381736546, -0.11101138379628155], [0.13387137644963198, 0.13221943751850887, -0.08855701042154071]], [[-0.4842909957633133, -0.16112703332780662, -0.13518698917560196], [-0.28532321664829285, 0.06881816384355052, 0.2998585594481757], [0.15675677812910843, -0.09781488629710154, -0.3928101202710947]]], [[[0.10803995368932565, 0.26719605381365535, 0.04900395047567288], [0.038576047122639755, 0.07176237375236753, -0.03174895413451701], [-0.25668193228582675, 0.42960228841766346, 0.2643436166223526]], [[0.32122328142658596, 0.002142917574923565, 0.17094115419365735], [-0.4770171177669494, -0.041609756482606494, -0.038187333534588704], [-0.20730704630752747, -0.2173760290386198, -0.41000198362990337]]]], "dtype": "float64"}, {"__nd__": [[0.5452422227788466, 1.0174700875309837], [0.04513420229427937, -0.06857211295303603], [0.16205023120012, 0.4297955007785647], [0.7394331205301121, 1.5217697505273675]], "dtype": "float64"}, {"__nd__": [[[[-1.8976890351109963]], [[1.2405317109173912]]], [[[1.0526410179783674]], [[0.042889246135958306]]], [[[0.5109452109910376]], [[-1.0610224204336542]]], [[[0.7999406167011613]], [[-0.7081330728060555]]], [[[-0.24155617025568174]], [[-0.3867988324257682]]], [[[0.6595592212341085]], [[0.1577593222917515]]], [[[0.1733925537512408]], [[0.7593364975514167]]], [[[-0.859271975423183]], [[0.34151789783306263]]]], "dtype": "float64"}, {"__nd__": [[0.6742933174129827, 1.1102052868303844, 1.234843442103481, 0.7160423876548735, 1.2127252667920145, 1.2442516396238439, 1.2975351530223704, 0.6563330296590143], [0.0647188631942162, -0.048888904315508436, -0.015073455148495243, 0.03469658915496961, 0.0015107192363865074, 0.031262240410611576, -0.058606989504497, 0.0034950310716132522], [0.21485210159508808, 0.15662568959342954, 0.08295993903018326, 0.07070060680365893, -0.3363547121871632, 0.23933580734517768, 0.33449932060805576, 0.26044902281830945], [1.0296260549270195, 0.5563609803114757, 0.5701100727596391, 0.9184059228141825, 1.000277801798288, 1.8426889473717862, 1.8123172985088565, 0.885655443231036]], "dtype": "float64"}, null, null, 1], "e": {"__tup__": [{"__nd__": [[[[0.8231574461546157, 0.0, 0.0, 0.5806433732421292, 0.8773715558784625], [0.4465767339859864, 0.0, 0.0, 0.5355763962581808, 1.4163116506570845], [0.0, 0.8922711872308136, 1.0506515880420895, 0.0, 0.0], [0.0, 0.0, 0.6128480077730941, 0.8212813112183021, 0.0], [0.0, 0.0, 0.0, 0.8907175976734721, 0.0]], [[0.0, 0.0, 0.0, 0.0, 1.6035930995440588], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.6296920625547521, 0.0, 0.0, 0.0], [0.0, 0.0, 0.2575412352782378, 0.4817570437667491, 0.0], [1.903320859145549, 0.04504814064092649, 0.0, 1.6656019206801385, 0.0]], [[0.0, 0.0, 0.0, 1.5995034924687397, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.362383333401255, 0.598675001366034, 2.11322253560913, 0.0], [0.022431257460434367, 0.10611756757359353, 0.35692896398725615, 0.5710710654072996, 0.0], [1.325228768288948, 0.0, 0.0, 0.0546948165816625, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.705519636735013], [0.45751604342870367, 0.0, 0.0, 0.15689183352232405, 0.35169840666234053], [0.0, 0.0, 0.03453881624103515, 0.0, 0.16429253379533387], [0.547961108979177, 0.0, 0.0, 0.5129106956895387, 0.24206177176142615], [0.0, 0.0, 0.6438423050821409, 0.5338893821186996, 0.0]], [[0.24385557950180467, 0.0, 0.6303399919024062, 1.4778157958732128, 0.8164720415819025], [0.0, 1.0335979747045836, 0.2679157226627195, 1.040499968683831, 0.0], [0.0, 1.1746291669973794, 1.5097375755509308, 0.0, 0.0], [2.258555749939745, 0.7743774897291087, 0.25623667973332875, 0.8983152555719834, 0.0], [0.0, 0.41917469485157915, 0.05985501088609613, 0.0, 0.4171490461713746]], [[0.0, 0.0, 0.0, 1.004229306915032, 1.2359182175338774], [0.0, 0.0, 0.2688425346081089, 0.6013107035010941, 0.0], [0.0, 1.6442436913215173, 0.0, 0.18147741988551835, 0.10007948979981018], [0.12162772061992383, 1.0784928608079376, 1.1908044888390976, 0.0, 0.0], [0.9032519723027511, 0.0, 0.0, 0.5914119368201816, 0.0]], [[0.0, 0.4440533496409709, 0.1844877900391061, 1.4997696396611062, 0.3065658030756019], [0.0, 0.652198596476409, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.1922407234754876, 0.0, 0.0, 0.0, 0.0], [0.0, 1.5068196632998418, 0.0, 0.37735950623094877, 0.29898110919240517]], [[0.0, 0.0, 0.0, 0.33967941153246084, 0.17694696452613287], [0.0, 0.7540789573252799, 0.7549779731692329, 0.0, 0.18256820091399703], [0.08100175978271859, 0.0, 0.0, 0.354947924651705, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.3975939508241807, 0.0, 0.0, 0.0]]], [[[1.0577283542396956, 0.0, 0.26644873401249836, 0.0, 0.0], [0.0, 0.0, 0.0, 0.5959864655858162, 0.0], [0.0, 0.8751088649060579, 0.0, 0.0, 0.8374126788032571], [0.9566992430382036, 0.0, 1.1315139188953869, 1.8119200530526842, 0.0], [1.0265435332678887, 0.0, 1.0047826914471052, 0.0, 0.6685466153667194]], [[0.974361171053661, 0.8555432506902894, 0.0, 0.0, 0.06353661301100005], [0.0, 0.0, 0.0, 0.3659253496010251, 0.0], [0.40569070811411756, 0.0, 0.0, 0.0, 1.9809633690742225], [0.07018853617116705, 0.19115556473864603, 1.358879271931607, 0.8351161152459186, 0.8201734063667332], [0.0, 0.0, 1.9537765728631127, 0.0, 0.0]], [[1.1105054286213205, 0.24172027519909167, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.08614493233394488], [0.0, 0.0, 0.7942221272338916, 0.0, 0.0], [0.0, 0.0, 0.3602532981156129, 0.0, 0.0], [0.7475753957751068, 0.0, 1.967148772741958, 1.004185369208899, 0.0]], [[2.114176897256216, 0.49447196564870544, 1.066155139154789, 0.0, 0.3635924224782759], [0.0, 0.3097492344445344, 0.0, 0.0, 0.0], [0.0, 0.0, 0.2520055530675651, 0.0, 0.0], [1.337479119082722, 0.0, 0.0, 0.35071247185848714, 0.587997750741923], [0.3542620474937559, 0.26957958819995576, 0.0, 0.0, 0.0]], [[2.0902312328819224, 0.0, 0.5151288828722913, 0.0, 0.42051114003427753], [0.0, 0.0, 0.6046810057153048, 2.17531771556181, 0.9090370585811486], [2.4273482621714364, 0.6796567538636098, 0.6190096390607689, 1.86288955591424, 0.4220300285896346], [0.0, 0.9242823070032847, 0.808223653204225, 0.5054384507876879, 0.8653795305147858], [0.06869691832878777, 0.3207081112543918, 0.0, 1.1574552319220717, 0.0]], [[1.0349939931832104, 0.0, 0.0, 0.5093414998613843, 0.19476984896486899], [0.29581799096309525, 0.0, 0.0, 1.3094341405567105, 0.0], [0.0, 1.0878248978508962, 1.5181639057579996, 0.0, 1.5729406083167121], [0.41333632494041517, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0450925859999352, 0.0, 0.6674237719007368, 0.8093341296735355]], [[0.0, 0.0, 0.09759008896331978, 0.0, 0.0], [0.12024849070995619, 0.0, 0.0, 0.1486927757510692, 0.28442321985905894], [0.0, 0.0, 0.17602195473008853, 0.5550722009231055, 0.26469955471667683], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.14401459041268283, 1.6170278648749694, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.5079270662414701, 0.42757265374172204], [0.0, 0.599701896481687, 0.0, 0.0, 0.5656702179971631], [0.0, 0.0, 0.0, 0.0, 0.5258642424329322], [1.2690994148702146, 0.0, 0.0, 0.0, 0.06062831583524586]]]], "dtype": "float64"}, 92]}}, {"a": [{"__nd__": [[[[0.6934308452332727, 0.06432077570196623, 1.6394416687531799, 1.153610758288094, 0.26563168878899407, -0.6244698959940954], [0.08083276384395123, -0.5325934528607646, -1.2438243730785363, 0.15908955672189606, 1.3072708793685819, -0.40861353807870987], [-0.9304059345481043, 0.08489415677092202, -0.6191263412509664, 0.9043327565245415, -0.24656983866211632, 0.07291946482880138], [-0.18751114239157274, 0.09801522804491189, -0.8479596849582208, 0.1468001612523542, -0.6895408727360928, -0.7100067197982055], [0.298368763702789, -1.6988108324341524, -0.7515841509890262, -0.2632159796819321, -0.2313184666433555, 0.47007840231376874], [1.1182776310621643, 0.05540956070180689, 0.02494299605655841, -0.12039241940394436, 0.04569206311047856, -0.08779995122495911]], [[2.937380215091906, -0.7223324769795089, 0.9342005584271255, 0.9290662596465812, 0.642617901554801, -0.10218202147444003], [0.5072893297538505, 1.0853741474038818, -0.8222248222056796, 1.181255633703482, 0.27646567647915016, 1.5324429877762804], [0.1927546673399824, 0.22530789023735348, 0.23989479320173585, 1.2878522576574338, -0.2972081292342978, 1.4777028666384235], [0.28318052060335624, -1.604684811069508, -0.25531338428988554, 1.435940325014245, -0.032685071211226824, -1.3366377647209207], [0.4718163712683525, -0.17539591593139653, 0.6676793084298822, -0.41145472290502616, 1.2054307080609967, 0.5911117369605953], [0.1286772836028721, -1.2888044685403366, -0.3759533861111273, 1.6365784836323176, 2.461146709602979, 0.8043173141485349]], [[-0.042016112094382156, 0.8933999609154643, -0.18157338656346106, -1.2520461748555531, -0.22333198288976036, -1.5462781325252568], [-0.11772065811962053, 0.21148232222233354, 0.8776394550154786, 2.14260677220417, -0.235549058241576, 1.0518031408984683], [-0.08785352779211818, 0.8981180144512744, 0.17291632690161923, 0.06145352859212328, -0.121603543681437, -0.8075142290743947], [-0.8923971538030516, 2.1207850162901942, -0.4519121018297987, -0.11412291050733635, -0.6990279672617925, -1.2399082771405883], [0.6202710712712861, -0.8065798273420606, -1.392757309819429, -1.1554835016672231, -0.15505130238960532, 0.8607157346438132], [-0.5249208377542135, 2.497771420453747, -1.254166088634639, 1.0243734377989138, -1.7396318218396247, -0.1663979122306143]], [[0.15475785063754208, -1.0757285175016162, 0.03504501565458935, 0.11347280421858012, -0.22673399807332145, 1.1841028066309227], [-0.18521659546045313, 1.3554921337301264, -0.994434009983197, 0.23443466175406727, -0.09827530324015175, -0.603698315559832], [0.30331558136556097, -0.6957701606806189, 0.5019523879504335, -1.2237596575105287, 0.004288239950843586, -0.46068147260745107], [1.5303953913408366, -0.017004907035539873, -1.1904680907114158, -2.182724298344211, -0.5535383131419016, -0.31858992187930024], [1.7100979575187198, 2.317853713202092, 0.0130372076512947, -0.33085945419234364, 1.3968191574396196, -1.1285423090279285], [-0.49136520922224924, 1.4554889508148847, -0.0443589757731309, 0.03992754799768231, 0.10061267091766511, -0.8511046847074574]]]], "dtype": "float64"}, {"__nd__": [[[[-0.3192493084259384]], [[-0.03297779459169658]], [[-0.8293961542031475]], [[-0.1951111010536114]]], [[[0.3519122746814355]], [[-0.09793568493943315]], [[-1.1917765247301852]], [[0.06447889518941946]]]], "dtype": "float64"}, {"__nd__": [[0.9367740268870574, 1.3886511872211433], [0.1376049378388327, 0.032499713668193736], [0.02907381608795781, -0.07749684671030807], [1.4146694195439877, 0.6460222621848358]], "dtype": "float64"}, {"__nd__": [[[[-0.1355946054658075, -0.43082596612383867, 0.14162819829924428], [-0.25501005410981115, -0.3809428002876225, 0.004035365994466534], [-0.18405620690192318, -0.19798008347737825, -0.11854270675989192]], [[0.38177991658465343, -0.15593161261721206, 0.09768058708392442], [0.05069138811282734, -0.031143363582035906, -0.05089049037177852], [-0.13568601330678032, 0.19365081811070328, 0.11193748606728861]]], [[[-0.2746522803141501, 0.39963000343063637, 0.32383131411348226], [0.3617788822341236, 0.41989426343955466, -0.17447993760362407], [0.03186559504615652, 0.09500466622548343, -0.17298997452895726]], [[0.06752040408871775, -0.1277431922376213, 0.33722649238142854], [-0.03229328809178187, -0.4023716969519669, -0.3368955282271928], [-0.2019034907259875, -0.17029806802325717, 0.2867176964439307]]]], "dtype": "float64"}, {"__nd__": [[1.4229570208397138, 1.444049639021023], [0.04862147762099511, 0.064182572047039], [0.2522359958705564, -0.18920883062601312], [0.9118889271639641, 0.9616777141432986]], "dtype": "float64"}, {"__nd__": [[[[0.26000099064747634]], [[0.8494090540367693]]], [[[-0.31850873179958744]], [[-0.949934607390809]]], [[[0.061169713063721064]], [[-0.9692590470732615]]], [[[-1.797269240284901]], [[0.2012020545286287]]], [[[0.6712351366602334]], [[-1.2132096919212227]]], [[[-0.004857349898046403]], [[-0.7344641118977371]]], [[[-0.4931691763924678]], [[0.04136645499388233]]], [[[-0.26838093042652733]], [[-0.7881700749398324]]]], "dtype": "float64"}, {"__nd__": [[0.7432865980529785, 1.0146218182430302, 0.6933262202585841, 1.3193109358079749, 1.228569132098086, 0.6093449413067588, 1.1780143965168541, 1.106176827655356], [0.05938212442041555, -0.0031848638465453057, -0.1440352771814144, -0.11494170167772444, -0.1437023086155232, 0.08036303114743903, 0.0558706179681317, -0.0748365211138629], [0.3730120693104172, -0.05099163328268639, 0.14853005058263558, 0.3113227143009214, -0.17246474963888786, -0.12931538619849947, -0.2066515033482459, -0.029599977123822202], [1.9189671278306881, 1.389177910136485, 1.1836425077929043, 0.6945596870617512, 1.3053545700999067, 1.6605264224537701, 1.6048096726643428, 0.9334094253661624]], "dtype": "float64"}, {"__nd__": [[[[-0.09791916837846773]], [[0.25663188309141627]], [[-0.6746940296352522]], [[-1.053784550886315]]], [[[-0.6207092960800215]], [[0.15920789089395235]], [[0.31592186149582285]], [[0.5580736968480835]]], [[[0.5442035254147863]], [[-1.178304007364206]], [[0.9901636844463457]], [[-0.5975188117891754]]], [[[0.6988916432767193]], [[0.05218324267494086]], [[-0.1909390300202858]], [[0.27341854636247176]]], [[[-0.3693670668563854]], [[-0.9696628431225065]], [[-0.13530934187997026]], [[-0.5624189863584279]]], [[[0.2674000420966199]], [[-0.018438730440396584]], [[-0.1172166096286196]], [[-0.42688264099197887]]], [[[-0.5631713701465229]], [[0.5034272818949512]], [[-0.0943895220322965]], [[0.3661555687069505]]], [[[0.3300635224196186]], [[-0.37525934080106216]], [[-0.13661953051238507]], [[0.60290662133037]]]], "dtype": "float64"}, {"__nd__": [[0.8305678728278019, 1.1366539622560698, 1.3010245863532777, 0.7044948576202903, 0.5373058991381199, 1.1045746976266917, 0.6292534403107312, 0.9369144760946648], [0.07048787505914887, 0.06489371053782046, -0.17708304130750574, -0.02427531376941953, -0.08262098050488478, -0.0033612336598416817, 0.11155413458132113, 0.04344786053764543], [0.17957952488418702, 0.0005555499571846164, -0.538202503127062, -0.5913918622464082, -0.0635389207537658, 0.14883657050588578, -0.06171762118628785, 0.0342019012561253], [0.8459767435022558, 1.7219282112748133, 0.6799004433369964, 1.4133193875871441, 1.604205869668522, 1.1920913353325393, 1.5226576273812107, 1.1801296973358435]], "dtype": "float64"}, 1], "e": {"__tup__": [{"__nd__": [[[[0.3406711210798257, 0.0738126082519732, 0.0, 0.5357002783268134, 0.24481832087195826, 0.0], [0.2457502297083971, 0.0, 1.0717981814662159, 0.0, 1.1562743278321101, 0.09247315780987633], [0.0, 0.0, 0.0, 1.5306354387949854, 0.0, 1.0339187967943284], [0.0, 0.0, 1.4763129354844615, 2.233915802966382, 0.7735822131932986, 0.5791745475097823], [0.0, 0.0, 0.8248891409784056, 1.2455749510759784, 0.0, 0.412483893940312], [0.6132008873685332, 0.0, 0.9344501018461803, 0.0, 1.2980137690456741, 1.2855776324943755]], [[0.06544785693839802, 0.0, 0.0, 0.0, 0.0, 0.5760428012873495], [0.0, 0.33675349781054986, 0.0, 0.8821461049176235, 0.0, 0.532149753526485], [0.6145085565001627, 0.0, 0.6257270299042309, 0.0, 0.16550312656747804, 0.0], [0.7405870616681455, 0.0, 0.0, 0.0, 0.012480701423298893, 0.0], [1.0061772367438084, 1.8436562291235907, 0.22646826535220646, 0.0, 0.0, 0.0], [0.0, 0.6660061361600651, 0.0, 0.6951714125291162, 0.0, 0.0]], [[0.0, 4.24152369480707, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.8837404539815372, 1.5002686331852588, 0.0, 0.0], [0.0, 2.1465263992756034, 0.0, 0.0, 0.5802316133475547, 0.0], [0.0, 6.509371245256019, 0.210345737192438, 0.0, 0.0, 0.6720314755278607], [0.0, 0.0, 0.0, 0.0, 0.0, 2.1467561413306346], [0.6250231835931751, 4.9398339169521375, 0.0, 0.0, 0.0, 0.0]], [[0.17893275258936936, 0.0, 0.45250350711052556, 0.384964296798216, 0.0, 0.0], [0.0, 0.0844599178365506, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.2193168210953919, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.06346286389110645, 0.0, 0.0, 0.04507498453206732, 0.48125186904681094, 0.0], [0.2542805035048833, 0.0, 0.16623895945140077, 0.0, 0.02657428760167846, 0.0]], [[0.0, 0.4785116842339514, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.15293329091060925, 0.0], [0.0, 0.0, 0.0, 0.0, 0.27976796550054, 0.7943459758683897], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0010806916668077982, 0.37707689764078145, 0.42005039621155477, 0.3817902766521356, 0.17185081253743564, 0.0], [0.0026352606804051076, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.20031541110028395, 0.0, 0.411240308502128, 0.0, 0.2744490511601865], [0.0, 0.0, 0.11364707424045753, 0.9566772012332427, 0.12359935995741123, 0.10506095818211682], [0.0, 0.0, 0.0, 0.0, 0.0, 0.4890259667366124], [0.4651273606610225, 0.0, 0.0, 0.0, 0.11687032575935241, 0.02171445261372848]], [[0.9832841102906285, 0.0, 0.17534936870136775, 0.3797247311805586, 0.44816441047802036, 0.8396923341236144], [0.4792748198348012, 1.1087314083232707, 0.39140801095301153, 0.5892691821172861, 0.0, 0.7384712367274989], [0.7751970021802042, 0.2513806585533427, 0.7222201500602838, 0.2657507828133088, 0.3922291691082565, 0.7023280017820386], [0.8463199839904161, 0.0, 0.3925463025325652, 0.3154756558044933, 0.5109892840476263, 0.2520197644567573], [0.7159525550579614, 1.2212215170019671, 0.8477868683644729, 0.3997052577719211, 1.1049753862023683, 0.15566371500077994], [0.047207622335099314, 0.22493338843760227, 0.37744142745028947, 0.8039292814425871, 1.1122874305538195, 0.506676789725242]], [[0.0, 0.0, 0.17696295687897434, 0.20723814227410242, 0.0, 0.6262102868345772], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.015562159118476374, 0.0], [0.728895518684635, 0.0, 0.0, 0.0, 0.0, 0.18391985726760596], [0.7213643693248042, 0.7899621890187655, 0.0, 0.0, 0.0, 0.0], [0.0, 0.18563775576686736, 0.0, 0.0, 0.0, 0.0]]]], "dtype": "float64"}, 132]}}, {"a": [{"__nd__": [[[[-0.19111048538815525, -1.5221372255649512, -1.047925504547072, 0.5509050869713639, -0.9176278600900687, 1.0885966428738538], [-0.7233878672914514, 1.5699638027697482, -0.7511867066562221, 0.9699145003114799, 0.0467920676132457, -0.10635776668788603], [0.29569457169544644, 0.9242297530911673, 0.2530613922289177, -0.6040695555110028, -0.002265071856873611, 0.22862488036387374], [-0.5424455383329835, 0.8305144849322065, -0.9643912818749472, -1.3908461182923004, -0.06986145501957758, -1.0823296949334398], [0.9213810399440316, 1.2283933338174888, -1.9199218519823251, 0.0424173755530267, -0.30297148871218627, 0.6972514080812208], [-0.19374139957297454, 0.825715914347132, -0.05092581781724962, 0.7098743756710321, -0.9867958591573955, 1.449169262880495], [-1.5946505849591295, -1.4568476998051991, 0.32708258261560125, 0.9644398464647502, 0.27090675611248244, -2.6652159845147287]], [[-1.3865383504178, 0.2455779447142241, 0.06523028914775313, 0.12412471393588302, -0.2938699276114278, -0.26697959590964637], [0.13816572643999184, 1.2594645494940522, 0.6779926665787442, -0.5075498700853823, 0.6025963788461354, 0.2775305515407654], [-0.9836982274023044, 0.8962817943288554, 0.11050381730477378, -0.3492406192912259, 0.3713385150272658, -2.9414453966946343], [1.6018501555876037, -0.31375640424785833, -0.44220973730339747, -0.9816311514449595, 0.3062766693148183, 0.7132523038096765], [1.1218877660677855, -0.6482441135538065, 1.949680440561103, -0.4865599659495324, 0.010171583959816875, 2.9709680247560915], [0.7807389292753778, -1.9831628375106127, 0.1812298459797691, 0.6112141407437528, -0.4037332728450959, 0.2940548298713008], [1.3554387910372043, -0.645350891773867, -0.4943700918213828, 0.7139834617281992, -0.38205094547502844, -0.22394582168094948]], [[0.4472087963008574, 1.0680366224756586, 1.0667312621203773, 2.111229106775675, 0.8928001100223784, 0.440979190401582], [-2.137595426084776, -0.001538573377309107, -0.6556754703166178, -1.6248083517428766, 0.02314361194268084, 0.45823680441617054], [-0.626947035175362, 0.7082387403962556, -1.1235981725804491, -0.7026558082914338, 1.1051366177091932, 0.5595276897399365], [0.48392276933076017, -0.34300898580410816, 0.7244430081044643, -1.2352527927094545, -0.055745584489954966, 0.08569443270444978], [-0.26980888471321146, 1.2719585377789535, 1.0944594237477954, 0.18767233573592848, 0.004210630267389292, 0.2315317514788769], [0.8536241678195915, 0.3428461754808744, 0.24331901689845267, 0.4315861654987565, 1.1104165823249634, 0.5473668122350783], [-0.4876408660949678, 0.5577390126678633, -0.9523491319721852, 1.2033689220062491, 2.287093067399515, 0.153212178587333]], [[0.9807662786072725, 1.2652012263523316, -0.5547460693570657, 0.8544364362247368, 1.3333178201309157, -0.26146237136493794], [0.8939447466124765, 0.5393257028062878, -0.9474541420555248, 0.8511086966958832, 1.8847929636701093, 0.24231652386340785], [0.4892232999470784, -0.8519030494407412, -0.9525532204489607, -0.45827792003673207, 0.3089848921759254, 0.41780020154993996], [0.4318534274663838, 0.9222222147764839, 0.5821749939242662, -0.1395109873705066, 0.831117042618287, -0.7039746136398898], [1.873114729373682, -0.7294390772874569, 0.04164654311714937, -0.9582239269811378, 1.4264138007028613, 0.5558668940998349], [-0.7786097534340028, -0.7940257239860249, 0.9583896753186018, -1.396142163868524, 0.252406540523139, -0.9657363731889724], [0.3770610659890955, 1.154226225279882, 0.027133975228425897, 0.540580431003188, 0.3851804919833047, -0.8401992826712548]]], [[[-0.42512102772877947, -0.738929603504043, -1.1797178015022778, -2.6657036966783108, 1.4356072899246437, -0.49149818943537066], [0.468943864159158, 0.46305247234653957, -0.6732829738441108, 0.4494542951952867, 0.6006955072389238, -0.3679524655077756], [-0.6248364630025226, 0.41569630340504954, 0.13297678739551397, -0.519830489763486, 0.8982344627190607, -1.3164219537236341], [1.0312918498575256, 0.10628421011682201, -1.1473210213260745, 0.41652318708120173, -0.7588562539776645, 0.03826719683416491], [-0.15688541779401088, -0.6990015299600253, 1.4016378834533436, -0.08112846177631369, -0.2010707480933866, 0.8807601799274012], [0.4938202027081077, -0.04028857519083193, 0.22751929778435565, -1.242202236734482, 1.3364770409057072, -1.5608032767561746], [1.5224410150831418, 0.5861697539156533, -0.41135051133699235, -0.7150157287633974, -1.7966757062783356, -1.5641100502182623]], [[1.406788388094788, 0.7288343896261986, 0.7648868010986116, 0.580616900741221, -1.5306834397434799, 0.6830989006728369], [-0.8719853507827428, -1.847411775722024, -0.6633324901400061, -0.5502601591899198, 1.2292487385542685, -1.1766969471984126], [-1.1168844389348331, -0.2231544294316544, 0.9783003839914927, -2.2504018976075932, -0.026549089428662527, 0.96005988150551], [1.7288378736301793, 0.4192456361213628, 0.7869210093399776, -0.2834038645434974, -0.253520434394717, 0.006409501992476277], [-0.9464609759546115, 0.11732052258962589, 0.2346795948480159, 0.9794695944655434, 1.7882970817301327, 0.339552552740875], [0.44077524318665084, 0.3802055920888044, -1.80286669670801, 0.47857718696565404, 0.034354953337043095, -0.884696325123955], [0.40980071885589925, -1.2382882246149562, 0.8156172425031544, -0.6123212554099238, -1.0886752272824747, -0.3575653817632154]], [[-0.10970030762054855, -1.2354500967060318, -2.0281107725700847, 1.2981054641056442, -1.2842617164623285, -0.6594243647407494], [-0.35577184013018764, -1.3108446320333675, -0.43655148781524317, 0.39149573619314193, 2.190258861225402, 1.455220048184258], [0.9997928709251331, -1.157912019243275, 0.9600888603250566, -0.47608702738947645, 1.902427954704686, -0.8372438615727928], [-1.1885597362273432, 0.2792993454484555, 1.2403120438121977, -0.872621129416759, -1.1538081909566764, 0.299708842948658], [-1.222969661347588, -0.3420041102717776, -1.105352656853267, -0.7955606661447415, 0.3323542585861701, 0.5915045983672953], [0.4628029095157075, -1.1798438605174473, 0.05416422793218626, 0.24310926321302273, -1.1842577677617472, -0.44641476757453796], [0.6097335618982069, 1.022508345558124, 1.2079266204746937, -0.9957178820671122, -0.802866030456322, -0.08736041033013481]], [[-0.5841209211766811, -1.159698664642225, -0.19880368817297425, 0.039243680943801186, 1.01660268601452, 0.0021594503321458584], [-1.183226721259719, 1.1383924138271486, -0.7801547678049546, 0.9778569127126636, -0.05609746559876727, -0.634129932938359], [1.9264453516857811, -0.42410884389346354, -2.2761278395178937, -0.9432900703009196, -0.6659168863884887, 0.8874530495381361], [-0.33722950530664747, -0.6951429961137092, -0.7948446318606174, -1.3862977693713623, -0.15828111583325852, 0.39392496901885515], [0.46371822958046244, -0.0986251848708865, -1.7092334806618548, -0.5752314844492608, 0.3363798476788993, 0.25007406949419725], [1.3591388193534755, 1.2648380213625852, 1.004539040928223, 1.777077137762033, 0.8857879467147496, 0.24294352435974398], [1.4859499895567356, -0.5768786486904326, 0.8962158786524965, -1.8457896966481895, -0.09838688804921418, 0.9826817931761376]]]], "dtype": "float64"}, {"__nd__": [[[[0.9195107789374725]], [[-0.9865371410670966]], [[-0.6097278899758994]], [[-0.11474877743889571]]], [[[-0.6516806475194802]], [[-0.23448958079071788]], [[0.018046509069910323]], [[0.7235123739694362]]], [[[-0.4461194941233281]], [[0.29797970022153886]], [[-0.775253511568549]], [[0.5780418045309292]]], [[[0.22794192697098573]], [[0.2798012229870672]], [[-0.17074435759304563]], [[0.6051155130390287]]]], "dtype": "float64"}, {"__nd__": [[1.303001053006299, 0.7089475610555024, 0.6043622711995219, 0.975574525272375], [0.05522165237109065, -0.016791764877383927, 0.028500443544777133, 0.22848272530895733], [-0.05366913174375483, -0.2850811883217958, 0.297982188495331, 0.12913620303150905], [0.500188323157542, 0.9300957585445201, 1.3190554688839284, 1.023982009707876]], "dtype": "float64"}, {"__nd__": [[[[0.08199132319887435, -0.08802313850962927, -0.2959825304722308], [-0.23984909714779581, -0.18322098330901274, 0.08740969108373575], [0.020060943022996026, -0.09217241417452103, 0.03534659331822638]], [[0.16422340042462416, 0.01408220450837991, 0.2037401054635263], [-0.16548730468723707, -0.027456224083540707, 0.14136563744670563], [-0.060660186743826706, -0.0243406883985271, -0.3833192874292277]], [[0.08303366292752692, 0.0854957216500337, -0.10384874760059903], [-0.08283045553575771, 0.021615163636072316, 0.16538412067146488], [-0.37320247965908243, -0.4360730537254179, 0.10937240899659258]], [[0.2571476887991494, 0.010360070062486326, -0.06697241889229986], [-0.030321680377793907, -0.0022172433627771887, 0.018390648350029494], [0.26806609508103046, -0.25789815840899244, 0.1221135137144098]]], [[[0.04141207158879771, 0.08465489904911232, -0.3637175257683725], [0.1435955614552221, -0.38727471540116426, -0.19973035143161835], [-0.13413868727836986, -0.07868651579213133, 0.015151595803497016]], [[-0.15182891303867133, 0.1159607485768702, 0.06780292647501153], [0.12665578969122557, 0.05289398059131268, -0.06157321970070117], [0.2104890713051908, -0.041745006340445846, -0.055032637176654325]], [[0.07600313869472007, 0.17838429316962923, 0.13744665530698472], [-0.05211467725750856, 0.142406771381115, 0.05425336862345204], [-0.30089376823879854, -0.0008333684031069164, -0.03536522788223267]], [[-0.08469496339601368, -0.08097347546408355, -0.10834984609676886], [-0.09105051715476749, 0.0015868754395436417, 0.1836122737298033], [-0.12565251011742198, 0.34444083653441004, -0.1677208986938826]]], [[[0.27060497636087033, -0.11097951211372625, -0.013961508492107408], [-0.08104656543557351, 0.3016175018786365, -0.06240518624064006], [0.0601804359331397, -0.22053211830922548, -0.05029343628340796]], [[-0.1816633240879084, -0.009176013965050689, -0.3366400195331554], [-0.30231313807969445, 0.23608878892005633, -0.03867606041065813], [-0.16082649423214282, 0.16784788188793354, -0.12403690067184182]], [[0.11838158465596656, -0.034615532518600174, -0.18610340506372622], [0.005508882183478566, -0.16293733448201778, -0.22387602571807472], [-0.443888534359042, 0.36663724880465437, -0.2946564317507338]], [[-0.17606974125791594, -0.3474914688783696, -0.3199956313347821], [0.14645890091785693, 0.2657714683522167, -0.1931160216940285], [-0.020137104597202703, 0.08305536638911067, 0.04580944451468138]]], [[[0.07631092145429251, 0.19492437269526156, -0.28224852832433184], [0.09271341259477116, 0.20339448702269744, 0.16161227055494812], [0.009892327744549245, 0.08087714202804822, 0.08009150391928242]], [[0.2267419224701017, -0.03317946921694331, -0.024009022825814397], [-0.3031407494035621, -0.00019470675342046805, -0.07177654146504923], [0.20467769309350256, -0.11716144364790304, 0.07398844224405442]], [[0.042119765787048825, -0.059496247538071916, 0.1608255249708178], [0.008006089683026503, 0.09514299156577001, -0.029053500745501965], [-0.05149821010653527, -0.29688642727873377, 0.05796747240735209]], [[-0.03232553622006795, -0.3168840117988885, -0.05926518646697943], [-0.09088926183606327, 0.141883594018831, -0.06932575621648507], [0.03818775627117483, -0.052004134246002275, -0.20292845003130278]]]], "dtype": "float64"}, {"__nd__": [[0.7507954683716564, 1.2689125608171135, 0.871126171294109, 1.4583850446943516], [-0.12080312728224535, -0.039776686430032675, 0.04874945575394011, 0.2593019040025343], [0.037298780146933684, 0.056026699449968735, 0.11243663338194512, -0.2360121301689303], [1.5437627182785276, 0.7958364311806418, 1.5264715267701083, 1.984364315311474]], "dtype": "float64"}, {"__nd__": [[[[0.2606421608128613]], [[0.4690566315251415]], [[0.07314404785066234]], [[-0.8108503629503939]]], [[[0.03597434069226717]], [[-1.0262083801704094]], [[0.2616702701170264]], [[0.1384982535505332]]], [[[-0.40798607047261515]], [[-0.43333978212446156]], [[-0.45471159043270704]], [[-0.44915759561316687]]], [[[0.1754678354061949]], [[0.2513066299458231]], [[-0.37136185417309686]], [[0.06888408628955967]]], [[[0.7099903612436669]], [[-0.20258136847245117]], [[0.1290123308986973]], [[-0.4697853562883895]]], [[[-1.0033433397199505]], [[-0.19899725200063617]], [[-0.3519602496166305]], [[0.6287569726011298]]], [[[-0.4031874653534658]], [[-0.055704456925681954]], [[-0.06114170037708571]], [[-0.13727510326371375]]], [[[1.0282875827682114]], [[0.0003595949552239695]], [[-0.5384632556664297]], [[-0.5331213200132204]]], [[[-0.3698499750454767]], [[-0.5194672722823974]], [[0.04123198493000763]], [[-0.3388389525939539]]], [[[-0.35328345333008393]], [[-0.5993884804995041]], [[0.3220831720047988]], [[-0.05160916541177862]]], [[[-0.32563650762053986]], [[0.3554059285104898]], [[0.30383118205697346]], [[0.6037032089817304]]], [[[-0.46518584065799173]], [[0.7146183685987412]], [[0.36331223689173403]], [[0.0896917663474347]]], [[[0.40682314147136445]], [[0.3540861457466882]], [[-0.470410790483359]], [[-0.66537765897262]]], [[[-0.8947917484519624]], [[-0.46042042581221965]], [[-0.4640964834684523]], [[0.30230787862376496]]], [[[-0.24708032596933535]], [[-0.41872179750764404]], [[0.2882797057530867]], [[0.1871614412511621]]], [[[0.18804507439390925]], [[0.7037477960420839]], [[-0.17844764005628505]], [[-0.14036182683586831]]]], "dtype": "float64"}, {"__nd__": [[0.6219478426253954, 0.7231442822934898, 1.4661343403484248, 0.511279360108282, 0.7747102617798708, 1.4974856777913472, 0.6261708706009322, 0.9307099375818915, 1.492055588140888, 0.8797601468354546, 0.7991406225511012, 0.6014269217456447, 1.3676777753853837, 1.0011308500297884, 1.4838693783273447, 1.121677711776157], [-0.09781771784031135, 0.05742835153908147, 0.08983065433440812, -0.09302676963311388, 0.13348839243668195, 0.15149152471674815, 0.05944330974591362, -0.11432582876590894, 0.009890558124166558, -0.050262138057200145, -0.0230913029325646, 0.003197353569898874, -0.026873882842690905, 0.02582069192286018, -0.05280998973921356, 0.14316899303527716], [-0.5053616607141017, -0.1287203151268274, 0.03235390705725926, -0.25699421159297003, -0.08253261414994034, -0.04874325255439562, -0.29932397287255075, -0.12585456957771687, -0.00804160858552643, -0.36768169641984433, -0.012644200006791324, 0.275494728840511, -0.0004954250346573055, 0.3015899841465775, 0.22794182591900208, -0.04286418367295905], [1.208653026823191, 1.0111461599240703, 1.3805417728255367, 0.620482105239889, 1.0098427743332734, 1.6626887854334467, 1.2537817624488454, 0.5228184320454261, 1.1488295802360482, 1.2788403726973476, 1.105132996818315, 1.9148815893729807, 1.8884447086934497, 0.9955655111536988, 0.9872684443552275, 1.4460974343877262]], "dtype": "float64"}, {"__nd__": [[[[-0.10900456991723387]], [[-0.5489587220591693]], [[-0.011520161183658615]], [[-1.1264784107342978]]], [[[0.4065789023064978]], [[0.45831699766381884]], [[-0.6278721072949732]], [[0.0738773687661952]]], [[[-0.8171539003919902]], [[-0.13656716989618403]], [[-0.3011759828976789]], [[0.0896457464516264]]], [[[0.6712849137341836]], [[0.5320694691644217]], [[0.056943145159869986]], [[0.29839276868036907]]], [[[-0.8805662699339332]], [[-0.22117742039492594]], [[-0.07229364797363427]], [[-0.09613038060596364]]], [[[0.1070860932015741]], [[0.26841670782264754]], [[-0.010784243113224625]], [[0.21139708830449408]]], [[[-0.29551068714291245]], [[-0.9113726475694235]], [[-0.27926294435833937]], [[0.2636935747560577]]], [[[0.4022661296293679]], [[-0.9429338462727214]], [[-0.713105828312945]], [[-0.3088422860053769]]], [[[-0.11740367936414682]], [[0.9908802555802143]], [[-0.22897964366357035]], [[0.773799755802753]]], [[[0.6588209037444176]], [[-0.4474461687800018]], [[0.43873909137991973]], [[0.32664502980354987]]], [[[-0.4304264214685511]], [[-0.12171356233467445]], [[-0.2726523928118334]], [[-0.15513046827818777]]], [[[-0.30243001504618383]], [[-0.5963510014910401]], [[-0.10477681688252602]], [[-0.31166498717970564]]], [[[0.14917436733985587]], [[-0.31461616150105437]], [[0.3942212362760858]], [[-0.29906525804699374]]], [[[0.10419442614635545]], [[-0.18383002880742313]], [[0.8316535077456555]], [[-0.009157967781256722]]], [[[0.1597244891845324]], [[-0.3076555839776411]], [[0.7437064842122146]], [[-1.1534098598514648]]], [[[-1.0245078677686263]], [[-0.5688301293768047]], [[-0.808218975077009]], [[-0.41261334100224434]]]], "dtype": "float64"}, {"__nd__": [[1.2878218959841568, 0.7549078481168652, 1.3252945078359941, 0.8103586730704987, 0.6736593635682417, 1.4373783012104517, 0.7472577633602383, 0.9636736429834173, 0.6238965003367208, 0.8403241306179952, 0.7958650198992251, 0.552524203168455, 0.609626889759167, 1.0195066664441286, 1.4281705438184562, 1.0020642859274127], [0.13589082994567486, -0.012282412902813221, 0.03822277156338582, 0.2450981740680241, -0.2213123860083294, -0.024887060889306138, 0.14983537367602542, -0.1647960374489101, 0.00992847508279874, 0.08983843272539938, -0.1721770740360204, -0.030003920139497958, 0.09121186841386074, 0.06295017417996623, 0.1486185995010375, -0.17130246120854423], [0.13827513639278965, -0.09130689176094373, -0.12104982328467136, 0.28950104733181964, -0.3475428776607218, 0.06766406449487634, 0.22981850555859742, -0.2793076308523214, 0.1494075066729774, -0.08668750719845618, -0.36923336426338704, 0.19613775310595508, -0.37253956340615874, 0.2767627377712468, -0.14238212626525745, -0.05446345022651235], [1.687943079678333, 0.9920862403804458, 1.2366644819176764, 0.6715873521560736, 0.6585848364597634, 0.9185208753104768, 1.2173426220065653, 1.620003717434732, 0.8919386033514214, 0.5486701443173727, 0.806691992858534, 1.7869687602505773, 0.7203661035287608, 1.425757590966143, 0.6838392184422888, 1.1421077688553127]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[[[0.0, 0.8093741597557522, 0.0], [0.0, 1.175883758557537, 0.0], [0.0, 0.0, 0.0], [0.0, 0.135236382483331, 0.0]], [[0.0, 0.0, 0.0], [0.3926530272730005, 0.8046332687880338, 0.0], [1.1767070923443896, 0.0, 0.0], [0.4395202572654297, 0.8352783802374056, 0.0]], [[0.12108869849102305, 0.7233196108349644, 0.4695156982916325], [0.0, 0.2690386999288567, 0.0], [0.0, 0.7941338531025052, 0.0], [1.7795885613169276, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.45553280734404866], [1.8001901187423146, 0.0, 0.3932289396566207], [0.0, 0.0, 0.24625523023895812]], [[0.6040888997498614, 0.9492103869390232, 0.7002422251601927], [0.0, 0.202472122731653, 0.0], [0.0, 0.9810455241579794, 0.0], [1.1802398082987475, 0.02431442227966968, 0.0]], [[0.0, 0.0, 0.3897202573541864], [0.3314476860926339, 0.0, 0.9438366467102111], [1.7406400607999872, 1.0195810597970072, 0.9047530146279568], [0.4987056335948633, 0.05760866676708154, 0.04367585547769148]], [[1.1746237687963057, 0.07919334729953406, 0.6029963992185016], [0.9131520087702168, 0.14455564189753545, 0.0], [0.0, 0.0, 0.45129923760713453], [0.0, 0.562206021353648, 0.03775484284136921]], [[0.0, 0.0, 0.0], [0.47637127709521276, 0.9219148921388487, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.15724478201514602], [0.0, 0.0, 0.0], [1.2956894674998352, 0.8549860443393459, 0.0], [1.2097445193121876, 0.0, 0.0]], [[1.7576503490231243, 0.0, 0.9199224358657974], [1.0379006256228498, 0.0, 0.7960082296304158], [1.077588177632539, 0.0, 0.40341968764835046], [0.0, 0.6661577333446942, 2.1093595661453186]], [[0.3267894866197587, 0.4152417134132812, 0.44904423838268825], [0.5942183892859723, 0.43548286461553865, 0.28721849292613183], [0.0, 0.8213753867813638, 0.5773385011363362], [0.669519318099764, 0.8205875356703773, 0.0]], [[0.11433697233204942, 0.0, 0.0], [0.0030993757299903335, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.06667688298645034, 0.0]], [[0.1424224072519129, 0.5363619902614418, 0.0], [0.0, 0.22101975077155409, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.9978688379811034]], [[0.0, 0.24953803727156418, 0.07263421261496783], [0.0, 0.0, 0.45416605701503054], [0.0, 0.0, 0.0], [0.0, 0.0, 1.2402525726769191]], [[0.0, 2.190813161978477, 0.0], [0.0, 0.46669881759442333, 0.8649316794128546], [0.0, 0.09956720575174904, 0.0], [0.0, 0.0, 2.462884668744592]], [[0.1329826446403602, 0.4231776045582502, 0.0], [0.4611067607081797, 0.9809004402563269, 0.0], [0.0, 0.0, 0.10001477726022096], [1.0955088432029854, 0.4787810378596674, 0.0]]], [[[0.0, 0.0, 0.0], [0.0, 1.261669042107013, 0.5590027213706192], [0.0, 1.5258878003860252, 0.0], [0.0, 0.0, 0.6521583800920104]], [[0.7036006983050043, 1.1496093215580374, 1.165937719984056], [0.0, 0.2243057660765671, 0.0], [0.5756535233088811, 1.2164459558513494, 0.46921974385199994], [0.6957958175846848, 0.0, 0.0]], [[0.0, 1.476022907960395, 0.0], [0.8691726004835797, 0.0, 0.0], [0.43627390176928826, 0.0, 0.0], [0.0, 0.0, 1.3894543033240074]], [[0.3656955257221594, 0.0, 0.21616177303823564], [0.0, 0.08098489838639006, 0.5564959349008959], [0.0, 0.549441499966161, 1.0930067651271387], [1.7694348370438382, 0.6846773967881041, 0.0]], [[0.0, 0.8090713558140691, 0.0], [0.7123627018463016, 0.0, 0.0], [0.3827063412391664, 0.0, 0.0], [0.0, 0.0, 1.35076406277901]], [[1.2726071817908555, 0.8044272718075492, 0.8360810296663163], [0.12636596107519585, 1.155086242979587, 0.4036445910667498], [0.1801336527252868, 0.3596508756412077, 1.8708703151041242], [1.570798215959034, 1.8946820303883587, 0.030085705820172448]], [[0.0, 0.26638431833442877, 1.1332708396837734], [1.1904140541772739, 0.0, 0.0], [1.086176539279323, 0.0, 0.0], [0.0, 0.0, 1.2476485685649046]], [[0.0, 0.0, 0.13184316639795357], [0.0, 0.0, 0.0], [0.551145122608497, 0.8968548416782078, 0.0], [0.0, 0.0, 0.04618419602621404]], [[0.038119121381939514, 0.319799258088749, 0.0], [0.08320296095741468, 0.0, 0.0], [0.0, 0.0, 0.23093302582454123], [0.350414915836172, 0.0, 0.0]], [[0.0, 0.0, 2.221548985532023], [1.735391600130725, 0.0, 1.7727191818274277], [0.4175539116652496, 0.142351665421537, 0.0], [2.173940797869922, 0.3359733043709835, 0.0]], [[0.8094527019709178, 1.4250498625107977, 0.9735871886538772], [0.0, 1.0058507052092438, 0.0], [0.9311215223255113, 0.3867910438297548, 0.7003586129837401], [0.0, 0.8024253831202115, 1.849002905965933]], [[0.0, 0.0, 0.13752503796227378], [0.0, 0.0, 0.0], [0.10420939743698689, 0.0, 0.0], [0.0, 0.0, 0.4957767786858174]], [[0.0, 0.0, 0.0], [0.3871354752284834, 0.0, 0.7299359310353999], [0.0, 0.04627356272396532, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.3644513886172568, 0.6477216523188284, 1.1591714856164428], [0.0, 0.0, 0.0], [0.29917974543533143, 0.595137258923031, 0.0]], [[0.49462859970635614, 0.0, 0.0], [0.0, 5.888190610087913, 4.183797382444157], [0.0, 2.432917747202588, 0.0], [0.0, 0.0, 0.0]], [[0.0, 2.288331496976435, 0.0], [0.0, 0.0, 0.0], [1.3150047889839083, 0.003748953876285391, 0.0], [0.0, 0.0, 3.1955695578633403]]]], "dtype": "float64"}, 368]}}, {"a": [{"__nd__": [[[[0.24598473417876918, 0.7155243745492311, 0.31136361480373376, 0.3626692597408993], [0.7222578583716579, 0.760049572763399, -0.46488446682318363, 0.5418962225755662], [1.5209969036865025, -0.40431935848489403, 2.213106965645032, -1.053531251193034], [-0.2759469110551827, 0.847257507024399, 0.125616624028224, 1.5807128730157067]], [[1.0573343240286577, -0.028942167576595155, 0.4695548743675972, -0.1184570623622354], [-1.2112673256089255, 1.9116370697247789, 0.05762818179601337, 0.5327285808583049], [-2.755430315025551, -0.9666861126559213, -0.5093043907618509, -0.42278807285062664], [0.14325476350013117, -0.44659147350572187, -1.462989948254052, 0.4669430812037274]], [[0.668748871741889, -0.30580183933120597, -0.24743143468553727, 0.2004020114513395], [0.8780347883850275, -0.9595992691326626, 1.4995898018925378, -0.7835356445983929], [-0.08539485606191517, 0.30216930240421097, 0.8105026632406613, -0.8883257607185443], [0.16308173260827746, -0.8294118561891602, 1.499634333335183, -0.06254747232143927]], [[1.508337495844989, 1.039313740204063, 0.3828728548295256, -0.23036427971774845], [0.3595101354560766, -0.34171176287874716, 0.3525105270264286, 1.094445577493796], [-1.0603115369558185, 1.1162198917710418, 0.14490366078742825, 0.22394491664525998], [0.6653387502289302, -0.14154287523151546, -0.14901958407993401, 0.19632019808951484]], [[-1.8851861281735236, -0.8355166130908509, -0.20279795814298768, 0.9377351767721177], [-0.8374294909829815, 0.6385728226527545, 0.12492169973285248, 0.46156865698094574], [0.04279348900801177, 0.3070953403323264, 0.2576027383851568, -0.8271039778395195], [3.250290340513973, -0.19153210952418534, 0.21993862164934616, 0.1838377266414732]], [[0.1632630729562186, -1.5419728005454274, -2.391817816080448, -1.1703927483714804], [-0.36263729692789365, 1.0515587853263018, 1.8410060756558417, 1.0539880617545154], [2.0696759007330052, 1.046376272526841, -0.6795795581568751, -0.7563943215836186], [0.34195154008229073, -0.46421506972272514, -0.21407007807485062, -1.29667856515558]]]], "dtype": "float64"}, {"__nd__": [[[[0.23463589447673766]], [[0.5651702233876538]], [[-1.0277772304554655]], [[0.20756585552761403]], [[-0.11363547137534777]], [[-0.5062656207119999]]], [[[0.06547823494988397]], [[-0.36388712857871935]], [[-0.03589623974101723]], [[0.011052187216003139]], [[0.2888007108397989]], [[0.37701337451019606]]], [[[0.530077888602363]], [[-0.08990329425950035]], [[-0.18770950018787458]], [[-0.6151377830051448]], [[-0.20619848922370965]], [[-0.23999894273025396]]]], "dtype": "float64"}, {"__nd__": [[1.103325467606336, 1.2701300847201968, 1.2248147788840165], [0.029294945652487095, -0.057912020351250926, 0.10998861819172366], [-0.007001828929548675, -0.12137214954623102, 0.04901358808093398], [1.094362632730864, 0.9008802251175139, 1.7417134210260241]], "dtype": "float64"}, {"__nd__": [[[[0.217040350992144, -0.014228864807232476, -0.21146688560854904], [-0.28063764335571145, 0.21285416894719955, -0.08491719484028674], [-0.10196811240199333, -0.18294176483395874, -0.12313349313608671]], [[-0.14748829766703586, 0.23683058891341385, 0.2713104246815779], [0.13922103984072184, -0.13852410945535754, -0.10797267426182004], [-0.06911591096939201, -0.187660419094634, 0.10270692804282994]], [[-0.19291513396756815, -0.03139338487365612, 0.00018041138376686288], [0.006055803941029208, -0.3375679480060773, 0.11615038903628191], [-0.41103626866855797, -0.024561916301139058, 0.02136685313228233]]], [[[0.030445489632291205, 0.09979912648037058, 0.2687453651022254], [-0.34849609044643765, -0.19062978354389382, -0.46105169630882403], [-0.1721749051723276, 0.12466887510901249, -0.08071423227659275]], [[0.07471155100971795, 0.06818529572205591, 0.45417550852233984], [-0.04452714743736627, -0.0950071226334972, -0.07017324888260108], [0.19680656231372995, -0.20502959474653076, -0.052027610171732734]], [[0.18558193743604118, -0.04971220007443909, -0.10130880366916706], [0.019087967234009597, -0.06788264139229859, 0.07574724245685535], [-0.12642504504804672, 0.2889718528257178, -0.2910046610254594]]], [[[-0.4129194562233183, -0.20510005534551057, 0.1482692909887756], [-0.0729299319127798, 0.3151868140229756, -0.20741769428892157], [0.08368610788670389, -0.011036857211995925, 0.27803760012037504]], [[0.018843550788027158, -0.01228382495250646, -0.06516760836262747], [0.1785325931468792, -0.07236134924252774, 0.2218379414458961], [0.016867176984886607, -0.15168752400679839, 0.06557761647565262]], [[-0.04511830141098872, -0.0772506586975197, 0.21584503794300955], [0.06394373826336489, -0.15466114596626154, 0.297393395418798], [-0.0814228111602088, -0.24983078057250432, 0.09462089783949258]]]], "dtype": "float64"}, {"__nd__": [[1.3767389778323706, 1.3453144628439717, 0.6095249800069121], [-0.11282484564311385, 0.13797790798235232, 0.0983166599228631], [-0.14722108299494738, 0.0006350354329177755, 0.13871969908305057], [0.5852244381523934, 1.221345958809648, 1.6182851843750143]], "dtype": "float64"}, {"__nd__": [[[[-0.5180500450709903]], [[-0.11763881443000264]], [[-0.42926116149263355]]], [[[0.7658942195813015]], [[0.06828499955783174]], [[0.5943503987010332]]], [[[-0.097092152643915]], [[0.021383960534982607]], [[0.8172907783022254]]], [[[-0.013941523793472414]], [[0.07355578876997665]], [[-0.7255865652289608]]], [[[1.0004411676287286]], [[-0.3803031564236566]], [[0.19447804514065448]]], [[[-0.12545442369472104]], [[-0.09797772792512957]], [[-0.6257861986936595]]]], "dtype": "float64"}, {"__nd__": [[1.413523409814111, 0.8267458217426482, 1.423089686483814, 0.933536073597576, 0.7319780883515281, 0.9216239718112468], [0.041437080389661825, 0.051461799136800206, -0.03156066872003283, -0.11812397880543635, -0.18035709978379766, -0.1643791321404885], [0.42292573443452197, -0.12045484768217699, -0.4262416857650959, -0.12988807705008257, 0.13525428809253243, 0.07982309602704768], [1.2112762751138342, 0.895011520380659, 1.7588818623069922, 1.5924526588770462, 0.9265308228222635, 0.6322164878387115]], "dtype": "float64"}, {"__nd__": [[[[-0.9342488162984695]], [[0.2775839602078594]], [[0.07518051160814364]], [[0.1672731672453551]], [[-0.05032330492650082]], [[0.14705349104482637]]], [[[-0.07435444585496967]], [[0.14113333047906337]], [[-0.15139436778979085]], [[0.3599896770030857]], [[-0.619255649933657]], [[0.29314503176306383]]], [[[-0.3524886378448738]], [[-0.8041724366562681]], [[-0.46396449573774756]], [[-0.28101968832914775]], [[0.3976606130267114]], [[0.5988737378637082]]], [[[0.07157595150304695]], [[-0.4266109803597067]], [[0.2916201342721189]], [[1.098172549343649]], [[-1.0527355500287672]], [[0.5619227342478283]]], [[[0.15140055269652347]], [[-0.5284965880438457]], [[-0.08518307161597886]], [[-0.3510326912163471]], [[-0.04813040324325842]], [[-0.10188901249240576]]], [[[0.11549443733802676]], [[-0.3200753273197333]], [[-0.5379368296255387]], [[0.03794973442263822]], [[0.012277309489783183]], [[0.22013023704069526]]]], "dtype": "float64"}, {"__nd__": [[0.619916941875963, 0.616572206020938, 0.6737972140430746, 1.2790619638580079, 0.9654957795118636, 0.944013219505697], [-0.1879689689329906, 0.08849246438912424, -0.07431023146715503, 0.0025089617805158127, 0.002181461146918508, 0.012624585746756664], [-0.35426848215991, -0.2741698511537453, -0.27573120911321686, -0.4191663902276888, -0.013831193621639674, -0.08368516892291639], [1.301214614143582, 1.4541832296365678, 1.469045929512606, 1.8061441450174773, 1.5014835762910526, 1.9772465662598262]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[[[0.0, 0.0], [0.0, 0.0]], [[1.3862719575228342, 0.4208206344188118], [0.3399592616825524, 0.04502341034985552]], [[0.0, 0.0], [2.498832061645774, 0.0]], [[3.6154540583727597, 0.0], [1.4273022943590923, 0.4918142486255237]], [[0.0, 0.0], [1.2193099555028217, 0.15627243357274623]], [[0.0, 0.0], [0.6879796368792772, 0.0]]]], "dtype": "float64"}, 189]}}, {"a": [{"__nd__": [[[[0.4889136218445176, -1.2152122143981312, 0.9030335999470479], [-1.227541837571259, -0.8061492055364032, -0.3852047724287589], [0.4848991627074963, -0.41770437296974516, 0.45400658471133226]], [[2.6646271761566664, -0.6805050087498529, 1.0918060460222905], [-1.6112881783214892, -0.24554176920850374, -0.6864802620281693], [-1.2117728080244345, 1.203830090109567, -0.2591373226393698]], [[-0.2881417354952528, -1.6071384661289116, -1.4302872807623177], [-1.4036363983327633, -0.5248170124089304, 0.3440489605018286], [-0.37348192309080186, -0.2872104374605628, 0.43879566941825066]], [[-0.8108862382420867, 0.005336306548240935, -0.3300354423050526], [-0.12913524361942644, 0.7253822817332543, 0.7893969965167158], [-0.23703970822085743, 1.1732692640572242, 1.6224767645839058]], [[-0.9495989663709378, 0.5134251398819574, 0.5040676622388246], [-1.152680202072859, 0.4949263251987994, 0.49130169432570714], [-2.070441205628761, 0.8422613503018056, -1.4758999797626604]], [[-0.8901574940088688, -1.9113169971352675, -0.33437784380768254], [-0.47482815454032834, 0.8722386017483627, -0.456269822910664], [-1.3764356569932525, 1.4179246863237782, -0.17544562093104327]], [[-0.30077734701528924, 0.915619509563576, -0.9888858803436297], [-1.8128583473880346, 0.25624135916930457, -0.26724201461013464], [-1.2274734452065654, -0.402447375889569, 0.6951013584685413]], [[1.3002431920495052, -2.2080813999555224, 0.10351709661524072], [0.20042270112432306, -1.282179024858797, -0.7299266145162631], [0.401071586558153, 0.11248844800905705, 1.5171936638060979]]], [[[-0.05625048201084952, 1.1620528540769146, -1.5830533667219024], [0.13167443691010336, -0.4747405033680282, -1.6676691158766208], [0.7157323355681248, 1.5018305915487915, 0.26352809902762414]], [[-1.788831792489087, 0.08597050894550069, -0.3165938211978186], [-1.4124858938839095, 0.8543019872079035, 1.298406362507045], [0.13837644517729103, 0.8985907488521024, 1.063619841197676]], [[0.7507938976090726, 1.4183677969932782, 1.0231570204164722], [0.6626274998624023, 1.6546343681043634, -0.8233789219191295], [2.005817505599068, -0.6204504431183595, 0.06315172329758202]], [[-0.2311952793517768, 0.6624128932586217, 1.2157474428338197], [0.7406304945626079, -0.21093014245835998, -1.0813395730005766], [2.3048660646932144, -1.3469252035434367, -0.08770667888806281]], [[0.7094663428109699, -0.5715884850485042, 1.3684211471429766], [-1.3708343894038677, 0.5250626068038062, -2.228431504880386], [1.9527140250279709, 2.6381081978195886, -0.980057329912257]], [[-0.271614842792202, 0.7141803109260354, -0.06256830085049889], [1.531330660276257, 0.811979836020118, -1.1985060739915576], [-0.8411993717519465, -0.7318062459532179, -0.46005793930948885]], [[0.019985433062277794, -0.10329987773929993, 0.8273008565191191], [0.9678579754973425, 0.9882716067948655, -1.7561089340835316], [-0.029074517243207715, 0.2534949938661429, -0.5133402487501648]], [[1.8344031362727713, 1.673601706020796, 0.49277226902549387], [-0.5848450713662628, -0.5493895018410723, 0.4707229053924656], [-0.4419580604595886, 0.09078033916616957, 0.2692806653550369]]], [[[0.4881858853933982, 2.254880203367974, -1.3602641699452729], [-1.5005815091496668, -0.095898988714525, -0.51171937537079], [-1.2149391329620167, 0.24449961072892556, -0.23208093171321434]], [[0.16978743967246618, -0.6839438761112243, -0.6634492247714113], [-2.0876530046167576, -0.697304907470666, -0.6779982846866087], [0.03002390819476346, -0.8261248345713843, 0.9288634833566236]], [[-0.021958009794567405, -0.2878284301077267, 1.733433255911393], [1.1394572275162402, 0.4025708736443483, -0.8080047972132645], [0.6093148365268635, 2.493740145911665, 0.707611359209942]], [[-0.39322338885548414, -0.966227338170226, -1.1197113738603792], [-1.2541718892026, -0.2938716105750459, -2.0616736815851953], [-1.3096447031598417, 0.19650414453895712, 0.806243257241963]], [[-1.0143631576714545, 1.0855808512151723, 1.1440053247678534], [0.4837956100868928, -0.5743128556753683, -0.5039026178419445], [0.8465922582869982, -0.94327037032086, -0.6749463493039072]], [[0.34634836947739367, 1.03946176107252, 0.5261312136559992], [1.860133147893219, 0.5382035278583829, 0.7095589788387534], [-0.4374868797901657, 1.7808032929606925, -1.1994033449534718]], [[0.2578565809724096, 0.0037106751720229775, 1.586052538666136], [-1.7977465411205649, 0.6933899662177532, 0.07299965338566987], [0.7408111913924128, -0.0667129717021593, -2.286343741607946]], [[-0.6706363124588123, -0.40704315956914777, -0.18183347912936043], [1.3550322881050172, -0.1466766180165353, -0.17491346219664855], [-0.5882591796338685, -0.32241414389257655, -0.9800948518380772]]]], "dtype": "float64"}, {"__nd__": [[[[0.6422011352699077]], [[0.4428085181521453]], [[-0.06852736364181548]], [[-0.6969384896936399]], [[0.21413608874003795]], [[-0.21983468629471936]], [[-0.14373691570457575]], [[0.10700502385899076]]], [[[-0.2096361439568953]], [[0.07691631337622944]], [[-0.2904267755433036]], [[0.07053593764584107]], [[0.33756720599854256]], [[0.08125186265817713]], [[-0.23322717666035186]], [[-0.20739237148715606]]], [[[-0.59490277007105]], [[-0.15660921770216002]], [[-0.36992975698029107]], [[-0.01204032026763237]], [[-0.45295992807819757]], [[0.227740727418205]], [[0.4129404042222896]], [[0.8117575985920789]]], [[[0.17658741897018015]], [[0.4083382354648136]], [[0.23326970547356876]], [[-0.16144922066447664]], [[-0.09873802567987883]], [[0.22415880805572957]], [[-0.20033031692037317]], [[-0.156221825510257]]]], "dtype": "float64"}, {"__nd__": [[0.6661491735238818, 0.6965965148787071, 1.0898293415502827, 1.0940657504676765], [-0.1769193020541496, -0.13990809134506385, 0.17209707578299177, -0.07921659482580624], [0.44678379536297497, -0.13033457219719968, -0.14650397006095417, 0.5166475354191219], [0.57617255518879, 1.8149744711806017, 1.1943929400186193, 1.9840661940779776]], "dtype": "float64"}, {"__nd__": [[[[-0.11216945667638552, 0.01020267702547043, -0.29727193651165734], [0.3068437942764027, -0.13057765038322555, 0.03582870570657443], [-0.041881787006740855, 0.4918750190306318, 0.13764231541123526]], [[0.13833146695454476, -0.1428961991460484, -0.3275335418790117], [0.27764956449453343, -0.1658917474470396, -0.020253030410965194], [-0.08610007466711178, 0.22016787505741522, -0.18559127808348005]], [[-0.16576275017808903, -0.026464523251423577, -0.21075076695830738], [0.11752934062002206, -0.10910928687133808, -0.36500777975758963], [-0.20670199079319104, 0.15360008868781164, -0.03692038797582229]], [[-0.03967225162243311, -0.3532182490109763, -0.00962180585691186], [-0.004910762264560341, -0.23871476604896213, 0.2079836778111289], [-0.12648024110938338, 0.13240653533292823, 0.20212768382375812]]], [[[0.166839262438926, 0.03571130147463787, 0.20364959400416396], [-0.0737546737762246, -0.14521955528489566, -0.12181929311533445], [0.11438711527197437, 0.03813961852508602, -0.09757892398828127]], [[-0.31038243046869857, 0.02174659300323363, -0.12721044300837409], [0.21373154954802018, -0.08462256773993075, 0.1201445435750663], [0.22052024277097995, -0.3683423831123496, -0.07804433739349236]], [[0.20336988310695567, -0.02650833559450221, 0.016429157813498376], [-0.12437302923461291, 0.050342467427304465, -0.047088148695491716], [-0.008149620221284326, 0.20512665137806393, -0.3323899425514191]], [[0.13833935726329358, 0.6262568793233232, -0.07783198636401642], [-0.14139948981094466, -0.2538087692362367, -0.1555080463178087], [0.15375909364197968, -0.08013500471138348, 0.016966202678085934]]], [[[-0.37099487294224415, 0.009056302007862829, -0.044541034049256904], [0.01600379638651171, 0.033994788151313494, 0.03413306931278881], [-0.1681717128048991, -0.002258293273537795, -0.16062133769162798]], [[-0.01411410614409486, 0.18855469315089293, -0.11618901786568288], [-0.18094664283857742, 0.23632477831891743, 0.28088391225966614], [-0.057485316576936535, 0.1239497563308754, 0.2419013674155502]], [[0.015453348859832503, 0.26441722025320435, -0.02847539188154387], [0.04421195442196143, -0.1667566790097034, -0.15773823268565088], [-0.02365288936855069, -0.18616795742408423, -0.11402815462801082]], [[-0.14790949720494204, 0.21216868309677517, -0.18455408413455374], [0.10666669153525292, -0.10478935294792895, -0.035148378497096025], [-0.08294845240664357, -0.18699611370953365, 0.33942732169670947]]], [[[0.07641758800049249, -0.13540277305064355, -0.23650172044387205], [-0.16837471386278668, 0.011998961096096104, -0.058185406185469135], [0.005201144533281044, -0.20726065677193614, -0.09077010051381072]], [[-0.03620375756057392, -0.15573102182233658, -0.01975633613678902], [-0.08507258217742623, 0.10151855664572323, -0.008077707349993747], [0.21159853187022662, 0.3039083789350265, -0.021494602106043842]], [[-0.1661158881911405, -0.0344173318403482, 0.18479786911363863], [0.10048799373156614, -0.09130401420557784, -0.08687727893753756], [-0.03959480775576812, -0.15699498118903576, -0.17366151452301418]], [[-0.19455089536237025, -0.04964340675666789, -0.18370273173309035], [0.07190209637260052, 0.16610882085678533, 0.12881983299327515], [-0.11195619172942751, 0.30621839495156505, -0.06294333245637974]]]], "dtype": "float64"}, {"__nd__": [[1.3200314343776633, 1.036846757167583, 0.8995371277194385, 0.6181285684320639], [0.0022798944371615988, -0.0065512906185056455, 0.061363272455300336, -0.16092683422830956], [0.24370208663078158, -0.12072797502448854, -0.061162894843615014, 0.3374441058294899], [0.788944581776784, 0.6874864845283184, 1.9941697115165846, 1.902197496813078]], "dtype": "float64"}, {"__nd__": [[[[-0.3023966503302549]], [[-0.1688728833858727]], [[-0.18716328688852787]], [[0.15897933572724252]]], [[[0.5484668175529889]], [[-0.45880101582136024]], [[-0.0182852115294434]], [[0.2616535339558637]]], [[[0.45912398358862805]], [[-0.4430322590802341]], [[-0.8424996751627233]], [[0.16860123344188438]]], [[[0.7747499124612792]], [[0.2545666389863754]], [[-0.15124828298427362]], [[-0.593682335766773]]], [[[0.7254148324218216]], [[-0.6871148783340852]], [[0.17209488558438715]], [[0.6550088301144901]]], [[[0.10097887347364418]], [[0.07770815169448426]], [[0.5372991459370591]], [[-0.7573470827588963]]], [[[-0.30245881133174046]], [[-0.3675617458966675]], [[0.5420936684952696]], [[0.44455639826366444]]], [[[-0.1769984910726279]], [[0.9754808595046109]], [[0.493872630525628]], [[-0.0666333686069056]]]], "dtype": "float64"}, {"__nd__": [[1.1491675841127251, 1.2298977801232471, 1.4392250982920212, 1.1330688331571537, 0.5200751335379233, 1.1635156062671106, 1.151912756091435, 0.5613489111201294], [0.059114140902433256, -0.043176138136464, -0.14504055212573272, -0.10189845976104034, -0.07390184197950918, -0.030841342491406194, 0.09905819576194802, -0.064600297514108], [-0.0280123979458771, -0.042057285415445005, 0.214370520016309, -0.03900632285761374, -0.10829494835271197, -0.07428886307594645, -0.06702562094984559, 0.10228956747428415], [0.551640685476035, 1.2886469878633622, 1.0157575356175228, 1.8921817268593435, 1.4692937304945972, 1.113894383543614, 1.6063296215775593, 0.9418262854917985]], "dtype": "float64"}, null, null, 1], "e": {"__tup__": [{"__nd__": [[[[0.540308899310757, 0.0, 0.9475285417688588], [0.0, 0.0, 0.0], [0.46309922852932756, 0.0, 0.5039817319579228]], [[2.569877710728083, 0.0, 1.0405700108729972], [0.0, 0.0, 0.0], [0.0, 1.0946794113090388, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.6556136747239913, 0.9550175203325664], [0.0, 1.1505532450030715, 1.5948254037323377]], [[0.0, 0.5074339102693675, 0.45395305158048915], [0.0, 0.46748868675874994, 0.29099841056493636], [0.0, 0.7486709541063281, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.9232950088491012, 0.0], [0.0, 1.4882041807164725, 0.0]], [[0.0, 1.2186990878632753, 0.0], [0.0, 0.4162170094745183, 0.0], [0.0, 0.0, 0.7879780220623637]], [[1.2867405296348526, 0.0, 0.06868917129413721], [0.3906035386870206, 0.0, 0.0], [0.438220212180482, 0.11533279418628102, 1.5067577691748995]]], [[[0.0, 1.2645082590008732, 0.0], [0.2238422785999903, 0.0, 0.0], [0.7123948078171296, 1.5433214549440035, 0.0]], [[0.0, 0.08836044247079129, 0.0], [0.0, 0.8261542526743141, 0.8678483234664961], [0.1335289375265304, 0.8575733072806438, 1.3920641132376017]], [[0.0009390581041964774, 0.9672037793516911, 0.22452177538007212], [0.16872275864562392, 1.0913836014526153, 0.0], [1.1151260324475518, 0.0, 0.0]], [[0.0, 0.5926442862493588, 1.2643716698583571], [0.6664359554667904, 0.0, 0.0], [2.1895830511521863, 0.0, 0.15538645282966623]], [[0.5428430040000188, 0.0, 1.1790884174511032], [0.0, 0.48483390979551705, 0.0], [1.9522514807655433, 2.5966217546309576, 0.0]], [[0.0, 0.765236718026774, 0.03629347804534645], [1.6034299375140686, 0.9049075645597717, 0.0], [0.0, 0.0, 0.0]], [[0.02224129871974515, 0.05667577256591379, 0.8014566598884487], [1.145336759294484, 1.1591786543897749, 0.0], [0.3108959268708562, 0.45344323740324244, 0.0]], [[1.9770268515508904, 1.5498348215370366, 0.6807203152622134], [0.0, 0.0, 0.8384010154781867], [0.0, 0.05224881118546282, 0.1912589636138539]]], [[[0.5678146326313148, 2.357335608291933, 0.0], [0.0, 0.0065564162094336464, 0.0], [0.0, 0.14578184916114734, 0.0]], [[0.1586638073131931, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.8019807169201403]], [[0.0, 0.0, 1.0217847415213277], [0.38124248204813016, 0.0, 0.0], [0.0, 1.4281034981193022, 0.0]], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.19759824584858565, 0.7440421219450708]], [[0.0, 1.058143212775123, 0.9952735567744593], [0.31327667328177183, 0.0, 0.0], [0.8198024163332773, 0.0, 0.0]], [[0.40404094989550515, 1.0905181681732585, 0.6155342465145142], [1.9527670248023492, 0.5892599349591214, 0.7807746948526442], [0.0, 2.0240315241802733, 0.0]], [[0.39516251894237886, 0.1636863254772367, 1.5989393077059706], [0.0, 0.8533656165229668, 0.15436273056699537], [1.0462032660218092, 0.055024575463380535, 0.0]], [[0.0, 0.0, 0.0], [1.5051118748076626, 0.0, 0.0], [0.0, 0.0, 0.0]]]], "dtype": "float64"}, 240]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(bottleneck_forward, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")